
# 3) KG Construction


The output of Notebook 2 is still a table of candidate facts. It is useful, but it is not yet a formal knowledge graph. To make the project interoperable with Semantic Web tools, we must convert these candidate facts into RDF triples with explicit URIs, classes, and properties.

This step is important because later modules depend on it:
- alignment requires stable entity identifiers;
- reasoning requires formal classes and properties;
- SPARQL requires a real RDF graph;
- embeddings benefit from normalized graph structure.



## RDF Reminder
An RDF triple has the form:

`subject - predicate - object`

In our project:
- subjects and objects are usually entities identified by URIs;
- predicates are ontology properties such as `won` or `playedAgainst`;
- some objects may also be literals, for example a year.

URIs are necessary because plain strings are ambiguous. For example, `Wimbledon` and `Wimbledon Championships` should not become two different nodes if they refer to the same concept.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from rdflib import Graph, RDF, RDFS

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.kg.kg_construction import (
    EX,
    ONTO,
    build_kg_artifacts,
    role_to_uri,
    serialize_graph,
    slugify,
)

TRIPLES_PATH = PROJECT_ROOT / "data" / "interim" / "extracted_knowledge.csv"
KG_DIR = PROJECT_ROOT / "kg_artifacts"
KG_DIR.mkdir(parents=True, exist_ok=True)
ONTOLOGY_PATH = KG_DIR / "ontology.ttl"
INITIAL_KG_PATH = KG_DIR / "initial_kg.ttl"
INITIAL_NT_PATH = KG_DIR / "initial_kg.nt"
STATS_PATH = KG_DIR / "graph_statistics.json"

print(f"Input triples: {TRIPLES_PATH}")
print(f"Ontology output: {ONTOLOGY_PATH}")
print(f"Initial KG output: {INITIAL_KG_PATH}")

Input triples: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/data/interim/extracted_knowledge.csv
Ontology output: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/kg_artifacts/ontology.ttl
Initial KG output: /Users/vincentlemeur/Documents/S8/DIA/Datamining/Project_datamining_bis/kg_artifacts/initial_kg.ttl



## Load the Extracted Candidate Triples
We begin with the CSV produced in Notebook 2. It contains candidate relations together with the sentence and source URL used as evidence.


In [ ]:
triples_df = pd.read_csv(TRIPLES_PATH)
print(f"Loaded {len(triples_df)} candidate triples.")
display(triples_df.head(10))

Loaded 722 candidate triples.


,subject,subject_type,subject_role,relation,object,object_type,object_role,sentence,source_url,strategy
0,Agassi,PERSON,Player,won,Wimbledon Championships,ORG,Tournament,"Following his Wimbledon loss, Nadal won 16 con...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
1,Albert Montañés,PERSON,Player,playedAgainst,Albert Portas,PERSON,Player,"In October, Nadal defeated No. 76 Albert Monta...",https://en.wikipedia.org/wiki/Rafael_Nadal,dependency
2,Alcaraz,PERSON,Player,participatedIn,French Open,ORG,Tournament,"In September 2020, aged 17, Alcaraz played his...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,cooccurrence
3,Alcaraz,PERSON,Player,participatedIn,Grand Slam,ORG,Tournament,"In doing so, Alcaraz also broke Nadal's record...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
4,Alcaraz,PERSON,Player,playedAgainst,Alex de Minaur,PERSON,Player,"At the Queen's Club, Alcaraz claimed his first...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
5,Alcaraz,PERSON,Player,playedAgainst,Alexander Zverev,PERSON,Player,Alcaraz defeated Alexander Zverev in another f...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
6,Alcaraz,PERSON,Player,playedAgainst,Botic van de Zandschulp,PERSON,Player,In his first main draw match at the Australian...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
7,Alcaraz,PERSON,Player,playedAgainst,Cameron Norrie,PERSON,Player,"At Indian Wells, Alcaraz defeated defending ch...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
8,Alcaraz,PERSON,Player,playedAgainst,Casper Ruud,PERSON,Player,"Two weeks later, at the Miami Open, Alcaraz de...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
9,Alcaraz,PERSON,Player,playedAgainst,Djokovic,PERSON,Player,The duo soon met for a rematch in the 2023 Wim...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency



## Ontology Design
We define a compact ontology that is rich enough for the project but still simple enough to present clearly.

### Core classes
At minimum, we use:
- `Player`
- `Tournament`
- `Match`
- `Surface`
- `Country`
- `TournamentEdition`

### Core properties
At minimum, we use:
- `won`
- `participatedIn`
- `playedAgainst`
- `hasSurface`
- `fromCountry`
- `editionYear`

This ontology is intentionally small. A compact ontology is easier to validate, explain, and align later.


In [ ]:
ontology_overview = pd.DataFrame(
    [
        {"element_type": "Class", "name": "Player", "description": "A tennis player"},
        {"element_type": "Class", "name": "Tournament", "description": "A Grand Slam tournament"},
        {"element_type": "Class", "name": "Match", "description": "A tennis match"},
        {"element_type": "Class", "name": "Surface", "description": "A playing surface such as clay or grass"},
        {"element_type": "Class", "name": "Country", "description": "A country associated with a player"},
        {"element_type": "Class", "name": "TournamentEdition", "description": "A tournament in a specific year"},
        {"element_type": "Property", "name": "won", "description": "Links a player to a tournament won"},
        {"element_type": "Property", "name": "participatedIn", "description": "Links a player to a tournament participated in"},
        {"element_type": "Property", "name": "playedAgainst", "description": "Links two opponent players"},
        {"element_type": "Property", "name": "hasSurface", "description": "Links a tournament to its surface"},
        {"element_type": "Property", "name": "fromCountry", "description": "Links a player to a country"},
        {"element_type": "Property", "name": "editionYear", "description": "Links a tournament edition to a year literal"},
    ]
)
display(ontology_overview)

,element_type,name,description
0,Class,Player,A tennis player
1,Class,Tournament,A Grand Slam tournament
2,Class,Match,A tennis match
3,Class,Surface,A playing surface such as clay or grass
4,Class,Country,A country associated with a player
5,Class,TournamentEdition,A tournament in a specific year
6,Property,won,Links a player to a tournament won
7,Property,participatedIn,Links a player to a tournament participated in
8,Property,playedAgainst,Links two opponent players
9,Property,hasSurface,Links a tournament to its surface



## URI Policy
A good RDF graph needs stable, readable, and predictable URIs.

We use a simple policy such as:
- `http://example.org/tennis/player/Rafael_Nadal`
- `http://example.org/tennis/tournament/Wimbledon_Championships`
- `http://example.org/tennis/surface/Grass`
- `http://example.org/tennis/country/Spain`

This policy has three advantages:
- it is easy to explain in a report;
- it separates entity categories cleanly;
- it reduces the risk of malformed identifiers.


In [ ]:
uri_examples = pd.DataFrame(
    {
        "label": ["Rafael Nadal", "Wimbledon Championships", "Grass", "Spain"],
        "role": ["Player", "Tournament", "Surface", "Country"],
    }
)
uri_examples["uri"] = uri_examples.apply(lambda row: role_to_uri(row["role"], row["label"]), axis=1)
display(uri_examples)
print("Slug example for 'Carlos Alcaraz':", slugify("Carlos Alcaraz"))

,label,role,uri
0,Rafael Nadal,Player,http://example.org/tennis/player/Rafael_Nadal
1,Wimbledon Championships,Tournament,http://example.org/tennis/tournament/Wimbledon...
2,Grass,Surface,http://example.org/tennis/surface/Grass
3,Spain,Country,http://example.org/tennis/country/Spain


Slug example for 'Carlos Alcaraz': Carlos_Alcaraz



## Build the Ontology and the Initial RDF Graph
The helper module does the following:
- cleans and normalizes the candidate triples again;
- removes obvious bad triples such as `subject == object`;
- creates typed entities and RDF predicates;
- adds provenance fields such as source URL and evidence text;
- exports the ontology and the initial graph.


In [ ]:
artifacts = build_kg_artifacts(triples_df)
serialize_graph(artifacts.ontology_graph, str(ONTOLOGY_PATH))
serialize_graph(artifacts.instance_graph, str(INITIAL_KG_PATH))
serialize_graph(artifacts.instance_graph, str(INITIAL_NT_PATH), fmt="nt")

with STATS_PATH.open("w", encoding="utf-8") as file:
    json.dump(artifacts.stats, file, indent=2)

print(f"Cleaned triples retained for RDF construction: {len(artifacts.cleaned_triples)}")
print(f"Ontology triples: {len(artifacts.ontology_graph)}")
print(f"Instance graph triples: {len(artifacts.instance_graph)}")

Cleaned triples retained for RDF construction: 640
Ontology triples: 41
Instance graph triples: 1787


/opt/anaconda3/lib/python3.13/site-packages/rdflib/plugins/serializers/nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(



## Inspect the Cleaned Triple Table Used for RDF
This table is important because the RDF graph should not be built directly from the raw extraction output. A second cleaning stage improves the graph quality before serialization.


In [ ]:
display(artifacts.cleaned_triples.head(20))

,subject,subject_type,subject_role,relation,object,object_type,object_role,sentence,source_url,strategy
0,Andre Agassi,PERSON,Player,won,Wimbledon Championships,ORG,Tournament,"Following his Wimbledon loss, Nadal won 16 con...",https://en.wikipedia.org/wiki/Rafael_Nadal,cooccurrence
1,Albert Montañés,PERSON,Player,playedAgainst,Albert Portas,PERSON,Player,"In October, Nadal defeated No. 76 Albert Monta...",https://en.wikipedia.org/wiki/Rafael_Nadal,dependency
2,Carlos Alcaraz,PERSON,Player,participatedIn,French Open,ORG,Tournament,"In September 2020, aged 17, Alcaraz played his...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,cooccurrence
3,Carlos Alcaraz,PERSON,Player,playedAgainst,Alex de Minaur,PERSON,Player,"At the Queen's Club, Alcaraz claimed his first...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
4,Carlos Alcaraz,PERSON,Player,playedAgainst,Alexander Zverev,PERSON,Player,Alcaraz defeated Alexander Zverev in another f...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
5,Carlos Alcaraz,PERSON,Player,playedAgainst,Botic van de Zandschulp,PERSON,Player,In his first main draw match at the Australian...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
6,Carlos Alcaraz,PERSON,Player,playedAgainst,Cameron Norrie,PERSON,Player,"At Indian Wells, Alcaraz defeated defending ch...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
7,Carlos Alcaraz,PERSON,Player,playedAgainst,Casper Ruud,PERSON,Player,"Two weeks later, at the Miami Open, Alcaraz de...",https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
8,Carlos Alcaraz,PERSON,Player,playedAgainst,Novak Djokovic,PERSON,Player,The duo soon met for a rematch in the 2023 Wim...,https://en.wikipedia.org/wiki/Carlos_Alcaraz,dependency
9,Carlos Alcaraz,PERSON,Player,playedAgainst,Novak Djokovic,PERSON,Player,They would meet again soon after in the 2023 W...,https://en.wikipedia.org/wiki/Novak_Djokovic,dependency



## Load the Serialized Graph Back with `rdflib`
A good validation habit is to reload the serialized graph from disk. This confirms that the Turtle export is syntactically valid and usable by downstream tools.


In [ ]:
kg_graph = Graph()
kg_graph.parse(INITIAL_KG_PATH, format="turtle")

ontology_graph = Graph()
ontology_graph.parse(ONTOLOGY_PATH, format="turtle")

print(f"Reloaded KG triples: {len(kg_graph)}")
print(f"Reloaded ontology triples: {len(ontology_graph)}")

Reloaded KG triples: 1787
Reloaded ontology triples: 41



## Sample RDF Triples
The next cell prints a small sample of the RDF graph. This helps us verify that the graph contains URIs, ontology predicates, and readable labels.


In [ ]:
for idx, triple in enumerate(kg_graph):
    print(triple)
    if idx >= 14:
        break

(rdflib.term.URIRef('http://example.org/tennis/player/Virginia_Wade'), rdflib.term.URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), rdflib.term.URIRef('http://example.org/tennis/ontology/Player'))
(rdflib.term.URIRef('http://example.org/tennis/tournamentedition/US_Open_1968'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'), rdflib.term.Literal('US Open 1968'))
(rdflib.term.URIRef('http://example.org/tennis/player/Felix_Auger_Aliassime'), rdflib.term.URIRef('http://example.org/tennis/ontology/playedAgainst'), rdflib.term.URIRef('http://example.org/tennis/player/Carlos_Alcaraz'))
(rdflib.term.URIRef('http://example.org/tennis/player/Medvedev'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label'), rdflib.term.Literal('Medvedev'))
(rdflib.term.URIRef('http://example.org/tennis/player/Carlos_Alcaraz'), rdflib.term.URIRef('http://example.org/tennis/provenance/evidenceText'), rdflib.term.Literal('"Alcaraz survives match point to beat Ramos Viñolas in m


## Graph Statistics
We compute a statistics summary to confirm that the initial KB is large enough for the next steps.

We especially check:
- total triples;
- total entities;
- total custom relations;
- class counts.


In [ ]:
with STATS_PATH.open("r", encoding="utf-8") as file:
    stats = json.load(file)

print(json.dumps(stats, indent=2))

{
  "total_triples": 1787,
  "total_entities": 236,
  "total_relations": 6,
  "predicate_uris": [
    "http://example.org/tennis/ontology/editionYear",
    "http://example.org/tennis/ontology/fromCountry",
    "http://example.org/tennis/ontology/hasSurface",
    "http://example.org/tennis/ontology/participatedIn",
    "http://example.org/tennis/ontology/playedAgainst",
    "http://example.org/tennis/ontology/won"
  ],
  "class_counts": {
    "Player": 180,
    "Tournament": 4,
    "Match": 0,
    "Surface": 3,
    "Country": 13,
    "TournamentEdition": 31
  }
}



## Class Counts and Predicate Inventory
These summaries make the graph easier to interpret before alignment.


In [ ]:
class_count_df = pd.DataFrame(
    [{"class_name": class_name, "count": count} for class_name, count in stats["class_counts"].items()]
)
display(class_count_df)

predicate_df = pd.DataFrame({"predicate_uri": stats["predicate_uris"]})
display(predicate_df)

,class_name,count
0,Player,180
1,Tournament,4
2,Match,0
3,Surface,3
4,Country,13
5,TournamentEdition,31


,predicate_uri
0,http://example.org/tennis/ontology/editionYear
1,http://example.org/tennis/ontology/fromCountry
2,http://example.org/tennis/ontology/hasSurface
3,http://example.org/tennis/ontology/participatedIn
4,http://example.org/tennis/ontology/playedAgainst
5,http://example.org/tennis/ontology/won



## Sanity Checks
Before moving to alignment, we verify that:
- duplicate RDF triples are not present;
- URI strings are well formed;
- the graph contains enough information to be useful;
- predicates are normalized and belong to our ontology.


In [ ]:
all_triples = list(kg_graph)
assert len(all_triples) == len(set(all_triples)), "Duplicate RDF triples detected."
assert all(str(s).startswith("http://example.org/") for s, _, _ in kg_graph if hasattr(s, "startswith") is False or True), "Malformed subject URI detected."
assert stats["total_triples"] >= 100, "The initial KG is too small."
assert stats["total_entities"] >= 50, "The graph has too few entities."
assert stats["total_relations"] >= 4, "Too few domain relations were created."

print("Basic graph-size checks passed.")

Basic graph-size checks passed.


In [ ]:
# A more explicit URI validation pass for URIRef subjects and predicates.
uri_validation_rows = []
for s, p, o in kg_graph:
    if str(s).startswith("http://example.org/") and str(p).startswith("http://"):
        uri_validation_rows.append(True)

assert all(uri_validation_rows), "Some RDF resources have malformed URIs."
print("URI validation passed.")

URI validation passed.



## Notes on Limitations
This initial graph is intentionally conservative.

Some limitations remain:
- match entities are still weakly represented because the extraction step mainly captured players and tournaments;
- some winner relations are still based on sentence-level heuristics and may require later correction;
- provenance is attached in a lightweight way and not yet fully reified.

These limitations are acceptable at this stage because the next notebooks will improve the graph through alignment, expansion, and reasoning.
